<a href="https://colab.research.google.com/github/hadi-hosseini/bandit/blob/main/Bandit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [128]:
import math
import cvxpy as cp
import numpy as np
from scipy.stats import ortho_group
from tqdm import tqdm
import random

# Parameters
k = 3
d = 1000
n_samples = 3000
sigma = 1.0
T = 50

orthogonal_matrix = ortho_group.rvs(dim=d)
mu = orthogonal_matrix[:k]
print(mu)

[[-0.03671601  0.00806821 -0.00074397 ...  0.06299988 -0.04118201
   0.04698379]
 [-0.02176344 -0.04456982 -0.08383259 ...  0.01272652  0.00800284
  -0.01204757]
 [ 0.00760438  0.02259875 -0.00537187 ... -0.01052249 -0.0262127
  -0.02560907]]


In [129]:
# Verify orthogonality
for i in range(k):
    for j in range(i+1, k):
        print(f"Dot product μ_{i+1}·μ_{j+1}: {np.dot(mu[i], mu[j])}")

Dot product μ_1·μ_2: 1.3877787807814457e-17
Dot product μ_1·μ_3: -4.206704429243757e-17
Dot product μ_2·μ_3: 7.719519468096792e-17


In [130]:
# create offline dataset
def create_logged_data(k, d, n_samples, sigma, mu):
  samples = []
  for i in range(k):
    # Generate samples from N(μ_i, σ²I)
    samples.append(np.random.normal(loc=mu[i], scale=sigma, size=(n_samples, d)))
  return samples

logged_data = create_logged_data(k, d, n_samples, sigma, mu)

In [101]:
# implement UCB algorithm
class UCBAlgorithm:
    def __init__(self, k, d, true_means, logged_data, perturbation):
        self.k = k
        self.d = d
        self.true_means = true_means
        self.logged_data = logged_data

        self.N = np.zeros(k)
        self.total_rewards = np.zeros(k)
        self.empirical_rewards = np.zeros(k)
        self.empirical_means = np.zeros((k, d))
        self.perturbation = perturbation

    def get_reward(self, x):
        return np.dot(self.true_means[0] + self.perturbation, x)

    def select_arm(self, t):
        if t < self.k:
            return t

        ucb_values = np.zeros(self.k)
        for j in range(self.k):
            mean_term = self.empirical_rewards[j]
            confidence_bound = math.sqrt((2 * math.log(t)) / self.N[j])

            ucb_values[j] = mean_term + confidence_bound

        return np.argmax(ucb_values)

    def update(self, arm, reward):
        self.N[arm] += 1
        self.total_rewards[arm] += reward
        self.empirical_rewards[arm] = self.total_rewards[arm] / self.N[arm]

    def run(self, T):
        rewards = np.zeros(T)
        chosen_arms = np.zeros(T, dtype=int)

        for t in tqdm(range(T)):
            arm = self.select_arm(t)
            sample = self.logged_data[arm][int(self.N[arm])]
            reward = self.get_reward(sample)

            self.update(arm, reward)
            self.empirical_means[arm] = self.empirical_means[arm] + (sample - self.empirical_means[arm])/self.N[arm]

            rewards[t] = reward
            chosen_arms[t] = arm

        return rewards, chosen_arms

In [5]:
# run UCB without perturbation
perturbation = 0.0
ucb = UCBAlgorithm(k, d, mu, logged_data, perturbation)
rewards, chosen_arms = ucb.run(T)
print("\nNumber of pulls per arm:", ucb.N)
print(chosen_arms)

100%|██████████| 50/50 [00:00<00:00, 24038.88it/s]


Number of pulls per arm: [42.  2.  6.]
[0 1 2 2 0 2 0 0 0 1 2 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 2 2 0]


In [6]:
def random_perturbation(d, epsilon):
    perturbation = np.random.randn(d)
    perturbation = epsilon * perturbation / np.linalg.norm(perturbation)
    return perturbation

epsilon = 0.5
random_perturbation = random_perturbation(d, epsilon)
ucb = UCBAlgorithm(k, d, mu, logged_data, random_perturbation)
rewards, chosen_arms = ucb.run(T)
print("\nNumber of pulls per arm:", ucb.N)
print(chosen_arms)

100%|██████████| 50/50 [00:00<00:00, 12631.92it/s]


Number of pulls per arm: [42.  2.  6.]
[0 1 2 2 2 0 2 0 0 0 1 0 2 2 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [103]:
# find adversary perturbation
# should learn this perturbation based on the logged data

class FindPerturbation:
    def __init__(self, k, d, true_means, logged_data, epsilon, qp=False, M=1, targeted=False, target_arm=1, infinity_attack=False):
        self.k = k
        self.d = d
        self.true_means = true_means
        self.logged_data = logged_data
        self.epsilon = epsilon
        self.qp = qp
        self.M = M # alternatives
        self.targeted = targeted
        self.infinity_attack = infinity_attack

        self.N = np.zeros(k)
        self.empirical_means = np.zeros((k, d))
        self.perturbation = None
        self.history = []
        self.all_perturbs = []
        self.target_arm = target_arm
        self.turn = 1

    def select_arm(self, t):
        if t < self.k:
            return t

        ### targetted
        if self.targeted:
          return self.target_arm

        ### untargetted
        else:
          self.turn += 1
          if self.turn == k:
            self.turn = 1
          return self.turn

    def find_perturbation_optimal_only(self, arm, t):
        x = cp.Variable(self.d)

        d_0 = self.empirical_means[arm] - self.empirical_means[0]
        c_0 = (math.sqrt(2 * math.log(t) / self.N[0]) - math.sqrt(2 * math.log(t) / self.N[arm])) - np.dot(self.true_means[0], d_0)
        self.history.append((d_0, c_0))


        constraints = []
        for (d_0, c_0) in self.history:
            constraints.append(x @ d_0 >= c_0 + 1e-6)
        constraints.append(cp.norm(x, 2) <= self.epsilon)
        prob = cp.Problem(cp.Minimize(0), constraints)
        prob.solve()

        if prob.status == 'optimal':
          self.all_perturbs.append(x.value)
          return x.value
        else:
          return None

    def find_perturbation(self, arm, t):
        x = cp.Variable(self.d)

        for j in range(self.k):
            if j != arm:
              d_j = self.empirical_means[arm] - self.empirical_means[j]
              c_j = (math.sqrt((2 * math.log(t)) / self.N[j]) - math.sqrt((2 * math.log(t)) / self.N[arm])) - np.dot(self.true_means[0], d_j)
              self.history.append((d_j, c_j))

        constraints = []
        for (d_j, c_j) in self.history:
            constraints.append(x @ d_j >= c_j + 1e-6)
        if qp:
          if self.infinity_attack:
            objective = cp.Minimize(cp.norm(x, "inf"))
          else:
            objective = cp.Minimize(cp.norm(x, 2))
          prob = cp.Problem(objective, constraints)
        else:
          if self.infinity_attack:
            constraints.append(cp.norm(x, "inf") <= self.epsilon)
          else:
            constraints.append(cp.norm(x, 2) <= self.epsilon)
          prob = cp.Problem(cp.Minimize(0), constraints)
        prob.solve()

        if prob.status == 'optimal':
          self.all_perturbs.append(x.value)
          return x.value
        else:
          return None


    def run(self, T, mode=1):
        chosen_arms = np.zeros(T, dtype=int)

        for t in tqdm(range(T)):
            arm = self.select_arm(t)

            if t >= self.k and ((t-k) % self.M == 0):
              if mode == 1: # check all inequalities
                perturbation = self.find_perturbation(arm, t)
              elif mode == 2: # check just optimal inequalities
                perturbation = self.find_perturbation_optimal_only(arm, t)


              if perturbation is None:
                return chosen_arms

              self.perturbation = perturbation

            sample = self.logged_data[arm][int(self.N[arm])]
            self.N[arm] += 1
            self.empirical_means[arm] = self.empirical_means[arm] + (sample - self.empirical_means[arm])/self.N[arm]

            chosen_arms[t] = arm

        return chosen_arms

## ABLATION 1 (CHECK ALL INEQUALITIES)

In [8]:
# check all inequalities (mode=1)
epsilon = 0.5
qp = False
find_perturbation = FindPerturbation(k, d, mu, logged_data, epsilon, qp)
chosen_arms = find_perturbation.run(T, mode=1)
print("\nNumber of pulls per arm:", find_perturbation.N)
print(chosen_arms)

100%|██████████| 50/50 [00:09<00:00,  5.49it/s]


Number of pulls per arm: [ 1. 24. 25.]
[0 1 2 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1
 2 1 2 1 2 1 2 1 2 1 2 1 2]


In [9]:
perturbation = find_perturbation.perturbation
norm_two = np.linalg.norm(perturbation)
norm_infinity = np.linalg.norm(perturbation, ord=np.inf)
print("norm 2:", norm_two)
print("norm infinity: ", norm_infinity)

0.3934783495774047


In [10]:
# all_perturbs = find_perturbation.all_perturbs

# for hist in all_perturbs:
#   ucb_with_perturb = UCBAlgorithm(k, d, mu, logged_data, hist)
#   rewards, chosen_arms = ucb_with_perturb.run(T)
#   print("\nNumber of pulls per arm:", ucb_with_perturb.N)
#   print(chosen_arms)

In [11]:
# run UCB with perturbation
ucb_with_perturb = UCBAlgorithm(k, d, mu, logged_data, perturbation)
rewards, chosen_arms = ucb_with_perturb.run(T)
print("\nNumber of pulls per arm:", ucb_with_perturb.N)
print(chosen_arms)

100%|██████████| 50/50 [00:00<00:00, 27249.90it/s]


Number of pulls per arm: [ 1. 24. 25.]
[0 1 2 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1
 2 1 2 1 2 1 2 1 2 1 2 1 2]


In [12]:
print(len(find_perturbation.history))

94


## ABLATION 2 (CHECK ONLY OPTIMAL INEQUALITIES)

In [13]:
# check just optimal inequalities (mode=2)
epsilon = 0.5
qp = False
find_perturbation = FindPerturbation(k, d, mu, logged_data, epsilon, qp)
chosen_arms = find_perturbation.run(T, mode=2)
print("\nNumber of pulls per arm:", find_perturbation.N)
print(chosen_arms)

100%|██████████| 50/50 [00:04<00:00, 12.18it/s]


Number of pulls per arm: [ 1. 24. 25.]
[0 1 2 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1
 2 1 2 1 2 1 2 1 2 1 2 1 2]


In [14]:
perturbation = find_perturbation.perturbation
norm_two = np.linalg.norm(perturbation)
norm_infinity = np.linalg.norm(perturbation, ord=np.inf)
print("norm 2:", norm_two)
print("norm infinity: ", norm_infinity)

0.19861657643608724


In [15]:
# run UCB with perturbation
ucb_with_perturb = UCBAlgorithm(k, d, mu, logged_data, perturbation)
rewards, chosen_arms = ucb_with_perturb.run(T)
print("\nNumber of pulls per arm:", ucb_with_perturb.N)
print(chosen_arms)

100%|██████████| 50/50 [00:00<00:00, 25643.82it/s]


Number of pulls per arm: [ 1. 19. 30.]
[0 1 2 2 1 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1]


In [16]:
print(len(find_perturbation.history))

47


## ABLATION 3 (QP)

In [17]:
# check all inequalities (mode=1)
epsilon = 0.5
qp = True
find_perturbation = FindPerturbation(k, d, mu, logged_data, epsilon, qp)
chosen_arms = find_perturbation.run(T, mode=1)
print("\nNumber of pulls per arm:", find_perturbation.N)
print(chosen_arms)

100%|██████████| 50/50 [00:18<00:00,  2.68it/s]


Number of pulls per arm: [ 1. 24. 25.]
[0 1 2 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1
 2 1 2 1 2 1 2 1 2 1 2 1 2]


In [18]:
perturbation = find_perturbation.perturbation
norm_two = np.linalg.norm(perturbation)
norm_infinity = np.linalg.norm(perturbation, ord=np.inf)
print("norm 2:", norm_two)
print("norm infinity: ", norm_infinity)

0.15484755563357896


In [19]:
# run UCB with perturbation
ucb_with_perturb = UCBAlgorithm(k, d, mu, logged_data, perturbation)
rewards, chosen_arms = ucb_with_perturb.run(T)
print("\nNumber of pulls per arm:", ucb_with_perturb.N)
print(chosen_arms)

100%|██████████| 50/50 [00:00<00:00, 26322.98it/s]


Number of pulls per arm: [ 1. 24. 25.]
[0 1 2 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1
 2 1 2 1 2 1 2 1 2 1 2 1 2]


In [20]:
print(len(find_perturbation.history))

94


## ABLATION 4 (M alternatives)

In [21]:
# check all inequalities (mode=1)
epsilon = 0.5
qp = False
alternatives = True
M = 10 # alternatives
find_perturbation = FindPerturbation(k, d, mu, logged_data, epsilon, qp, M)
chosen_arms = find_perturbation.run(T, mode=1)
print("\nNumber of pulls per arm:", find_perturbation.N)
print(chosen_arms)

100%|██████████| 50/50 [00:00<00:00, 458.09it/s]


Number of pulls per arm: [ 1. 24. 25.]
[0 1 2 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1
 2 1 2 1 2 1 2 1 2 1 2 1 2]


In [22]:
perturbation = find_perturbation.perturbation
norm_two = np.linalg.norm(perturbation)
norm_infinity = np.linalg.norm(perturbation, ord=np.inf)
print("norm 2:", norm_two)
print("norm infinity: ", norm_infinity)

0.17166878076050043


In [23]:
# run UCB with perturbation
ucb_with_perturb = UCBAlgorithm(k, d, mu, logged_data, perturbation)
rewards, chosen_arms = ucb_with_perturb.run(T)
print("\nNumber of pulls per arm:", ucb_with_perturb.N)
print(chosen_arms)

100%|██████████| 50/50 [00:00<00:00, 27806.31it/s]


Number of pulls per arm: [ 1.  1. 48.]
[0 1 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2]


In [24]:
print(len(find_perturbation.history))

10


## Ablation 5 (Targeted Attack)

In [135]:
# check all inequalities (mode=1)
epsilon = 0.5
qp = False
find_perturbation = FindPerturbation(k, d, mu, logged_data, epsilon, qp, targeted=True, target_arm=2)
chosen_arms = find_perturbation.run(T, mode=1)
print("\nNumber of pulls per arm:", find_perturbation.N)
print(chosen_arms)

100%|██████████| 50/50 [00:09<00:00,  5.39it/s]


Number of pulls per arm: [ 1.  1. 48.]
[0 1 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2]


In [136]:
perturbation = find_perturbation.perturbation
norm_two = np.linalg.norm(perturbation)
norm_infinity = np.linalg.norm(perturbation, ord=np.inf)
print("norm 2:", norm_two)
print("norm infinity: ", norm_infinity)

norm 2: 0.23112302658295145
norm infinity:  0.02558483599660052


In [137]:
# run UCB with perturbation
ucb_with_perturb = UCBAlgorithm(k, d, mu, logged_data, perturbation)
rewards, chosen_arms = ucb_with_perturb.run(T)
print("\nNumber of pulls per arm:", ucb_with_perturb.N)
print(chosen_arms)

100%|██████████| 50/50 [00:00<00:00, 20915.05it/s]


Number of pulls per arm: [ 1.  1. 48.]
[0 1 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2]


In [138]:
print(len(find_perturbation.history))

94


## ABLATION 6 (infinity norm)

In [77]:
epsilon = 1/125
qp = False
find_perturbation = FindPerturbation(k, d, mu, logged_data, epsilon, qp, targeted=True, target_arm=2, infinity_attack=True)
chosen_arms = find_perturbation.run(T, mode=1)
print("\nNumber of pulls per arm:", find_perturbation.N)
print(chosen_arms)

100%|██████████| 50/50 [00:08<00:00,  5.92it/s]


Number of pulls per arm: [ 1.  1. 48.]
[0 1 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2]


In [78]:
perturbation = find_perturbation.perturbation
norm_two = np.linalg.norm(perturbation)
norm_infinity = np.linalg.norm(perturbation, ord=np.inf)
print("norm 2:", norm_two)
print("norm infinity: ", norm_infinity)

norm 2: 0.15318620002320246
norm infinity:  0.0070774087049281075


In [79]:
# run UCB with perturbation
ucb_with_perturb = UCBAlgorithm(k, d, mu, logged_data, perturbation)
rewards, chosen_arms = ucb_with_perturb.run(T)
print("\nNumber of pulls per arm:", ucb_with_perturb.N)
print(chosen_arms)

100%|██████████| 50/50 [00:00<00:00, 23175.51it/s]


Number of pulls per arm: [ 1.  1. 48.]
[0 1 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2]


In [80]:
print(len(find_perturbation.history))

94


## ABLATION 7 (Restricted Threat Model: Infer Mus)

In [131]:
empirical_mus = [np.mean(arm_samples, axis=0) for arm_samples in logged_data]
error = np.linalg.norm(empirical_mus[0] - mu[0])
print("L2 distance:", error)

L2 distance: 0.5901347368602353


In [132]:
# check all inequalities (mode=1)
epsilon = 0.5
qp = False
find_perturbation = FindPerturbation(k, d, empirical_mus, logged_data, epsilon, qp, targeted=True, target_arm=2)
chosen_arms = find_perturbation.run(T, mode=1)
print("\nNumber of pulls per arm:", find_perturbation.N)
print(chosen_arms)

100%|██████████| 50/50 [00:15<00:00,  3.24it/s]


Number of pulls per arm: [ 1.  1. 48.]
[0 1 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2]


In [133]:
perturbation = find_perturbation.perturbation
norm_two = np.linalg.norm(perturbation)
norm_infinity = np.linalg.norm(perturbation, ord=np.inf)
print("norm 2:", norm_two)
print("norm infinity: ", norm_infinity)

norm 2: 0.22319735638068672
norm infinity:  0.027100961051095225


In [134]:
# run UCB with perturbation
ucb_with_perturb = UCBAlgorithm(k, d, mu, logged_data, perturbation)
rewards, chosen_arms = ucb_with_perturb.run(T)
print("\nNumber of pulls per arm:", ucb_with_perturb.N)
print(chosen_arms)

100%|██████████| 50/50 [00:00<00:00, 25735.08it/s]


Number of pulls per arm: [ 1.  1. 48.]
[0 1 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2]


In [127]:
print(len(find_perturbation.history))

94


## ETC Attack

In [34]:
# implement UCB algorithm
class ETCAlgorithm:
    def __init__(self, k, m, d, true_means, logged_data, perturbation):
        self.k = k
        self.m = m
        self.d = d
        self.true_means = true_means
        self.logged_data = logged_data

        self.N = np.zeros(k)
        self.total_rewards = np.zeros(k)
        self.empirical_rewards = np.zeros(k)
        self.empirical_means = np.zeros((k, d))
        self.perturbation = perturbation

    def get_reward(self, x):
        return np.dot(self.true_means[0] + self.perturbation, x)

    def select_arm(self, t):
        if t < self.k * self.m:
            return t % k

        return np.argmax(self.empirical_rewards)

    def update(self, arm, reward):
        self.total_rewards[arm] += reward
        self.empirical_rewards[arm] = self.total_rewards[arm] / self.N[arm]

    def run(self, T):
        rewards = np.zeros(T)
        chosen_arms = np.zeros(T, dtype=int)

        for t in tqdm(range(T)):
            arm = self.select_arm(t)
            sample = self.logged_data[arm][int(self.N[arm])]
            reward = self.get_reward(sample)
            self.N[arm] += 1

            if t < self.k * self.m:
              self.update(arm, reward)
              self.empirical_means[arm] = self.empirical_means[arm] + (sample - self.empirical_means[arm])/self.N[arm]

            rewards[t] = reward
            chosen_arms[t] = arm

        return rewards, chosen_arms

In [35]:
# run ETC without perturbation
perturbation = 0.0
m = 5
etc = ETCAlgorithm(k, m, d, mu, logged_data, perturbation)
rewards, chosen_arms = etc.run(T)
print("\nNumber of pulls per arm:", etc.N)
print(chosen_arms)

100%|██████████| 50/50 [00:00<00:00, 45022.58it/s]


Number of pulls per arm: [40.  5.  5.]
[0 1 2 0 1 2 0 1 2 0 1 2 0 1 2 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [36]:
# find adversary perturbation
# should learn this perturbation based on the logged data

class FindPerturbationETC:
    def __init__(self, k, m, d, target_arm, true_means, logged_data, epsilon, qp=False):
        self.k = k
        self.m = m
        self.d = d
        self.true_means = true_means
        self.logged_data = logged_data
        self.epsilon = epsilon
        self.qp = qp
        self.target_arm = target_arm

        self.N = np.zeros(k)
        self.empirical_means = np.zeros((k, d))
        self.perturbation = None

    def select_arm(self, t):
        if t < self.k * self.m:
            return t % k

        return self.target_arm

    def find_perturbation_with_l2_ball(self, arm, t):
        x = cp.Variable(self.d)
        constraints = []

        for j in range(self.k):
            if j != arm:
              d_j = self.empirical_means[arm] - self.empirical_means[j]
              c_j = - np.dot(self.true_means[0], d_j)
              constraints.append(x @ d_j >= c_j + 1e-6)

        if qp:
          objective = cp.Minimize(cp.norm(x, 2))
          prob = cp.Problem(objective, constraints)
        else:
          constraints.append(cp.norm(x, 2) <= self.epsilon)
          prob = cp.Problem(cp.Minimize(0), constraints)
        prob.solve()

        if prob.status == 'optimal':
          return x.value
        else:
          return None


    def run(self, T):
        chosen_arms = np.zeros(T, dtype=int)

        for t in tqdm(range(T)):
            arm = self.select_arm(t)

            if t == self.m * self.k:
              self.perturbation = self.find_perturbation_with_l2_ball(arm, t)
              return chosen_arms

            sample = self.logged_data[arm][int(self.N[arm])]
            self.N[arm] += 1
            self.empirical_means[arm] = self.empirical_means[arm] + (sample - self.empirical_means[arm])/self.N[arm]

            chosen_arms[t] = arm

        return chosen_arms

In [37]:
# check all inequalities (mode=1)
epsilon = 0.5
qp = False
target_arm = 2
m = 5
find_perturbation = FindPerturbationETC(k, m, d, target_arm, mu, logged_data, epsilon, qp)
chosen_arms = find_perturbation.run(T)
print("\nNumber of pulls per arm:", find_perturbation.N)
print(chosen_arms)

 30%|███       | 15/50 [00:00<00:00, 507.56it/s]


Number of pulls per arm: [5. 5. 5.]
[0 1 2 0 1 2 0 1 2 0 1 2 0 1 2 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [38]:
perturbation = find_perturbation.perturbation
norm = np.linalg.norm(perturbation)
print(norm)

0.053941175853323135


In [39]:
# run ETC with perturbation
m = 5
etc = ETCAlgorithm(k, m, d, mu, logged_data, perturbation)
rewards, chosen_arms = etc.run(T)
print("\nNumber of pulls per arm:", etc.N)
print(chosen_arms)

100%|██████████| 50/50 [00:00<00:00, 47053.00it/s]


Number of pulls per arm: [ 5.  5. 40.]
[0 1 2 0 1 2 0 1 2 0 1 2 0 1 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2]


## epsilon-greedy Attack

In [40]:
# implement epsilon_greedy algorithm
class EpsilonGreedyAlgorithm:
    def __init__(self, k, d, epsilon, true_means, logged_data, perturbation):
        self.k = k
        self.d = d
        self.true_means = true_means
        self.logged_data = logged_data
        self.epsilon = epsilon

        self.N = np.zeros(k)
        self.total_rewards = np.zeros(k)
        self.empirical_rewards = np.zeros(k)
        self.perturbation = perturbation

    def get_reward(self, x):
        return np.dot(self.true_means[0] + self.perturbation, x)

    def select_arm(self, t):
        if t < self.k:
            return t

        else:
          if random.random() < self.epsilon:
            return random.randint(0, k - 1)
          else:
            return np.argmax(self.empirical_rewards)

    def update(self, arm, reward):
        self.N[arm] += 1
        self.total_rewards[arm] += reward
        self.empirical_rewards[arm] = self.total_rewards[arm] / self.N[arm]

    def run(self, T):
        rewards = np.zeros(T)
        chosen_arms = np.zeros(T, dtype=int)

        for t in tqdm(range(T)):
            arm = self.select_arm(t)
            sample = self.logged_data[arm][int(self.N[arm])]
            reward = self.get_reward(sample)
            self.update(arm, reward)

            rewards[t] = reward
            chosen_arms[t] = arm

        return rewards, chosen_arms

In [41]:
# run epsilon_greedy without perturbation
perturbation = 0.0
epsilon = 0.1
random.seed(42)
epsilon_greedy = EpsilonGreedyAlgorithm(k, d, epsilon, mu, logged_data, perturbation)
rewards, chosen_arms = epsilon_greedy.run(T)
print("\nNumber of pulls per arm:", epsilon_greedy.N)
print(chosen_arms)

100%|██████████| 50/50 [00:00<00:00, 47608.44it/s]


Number of pulls per arm: [43.  3.  4.]
[0 1 2 2 1 2 0 2 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [42]:
# find adversary perturbation
# should learn this perturbation based on the logged data

class FindPerturbationEpsilonGreedyAlgorithm:
    def __init__(self, k, d, epsilon, target_arm, true_means, logged_data, epsilon_attack, qp=False):
        self.k = k
        self.d = d
        self.true_means = true_means
        self.logged_data = logged_data
        self.epsilon_attack = epsilon_attack
        self.qp = qp
        self.target_arm = target_arm
        self.epsilon = epsilon

        self.N = np.zeros(k)
        self.empirical_means = np.zeros((k, d))
        self.perturbation = None
        self.is_uniform = False
        self.history = []

    def select_arm(self, t):
        if t < self.k:
            return t

        else:
          if random.random() < self.epsilon:
            self.is_uniform = True
            return random.randint(0, k - 1)
          else:
            self.is_uniform = False
            return self.target_arm

    def find_perturbation_with_l2_ball(self, arm, t):
        x = cp.Variable(self.d)
        constraints = []

        for j in range(self.k):
            if j != arm:
              d_j = self.empirical_means[arm] - self.empirical_means[j]
              c_j = - np.dot(self.true_means[0], d_j)
              self.history.append((d_j, c_j))


        for (d_j, c_j) in self.history:
            constraints.append(x @ d_j >= c_j + 1e-6)

        if qp:
          objective = cp.Minimize(cp.norm(x, 2))
          prob = cp.Problem(objective, constraints)
        else:
          constraints.append(cp.norm(x, 2) <= self.epsilon_attack)
          prob = cp.Problem(cp.Minimize(0), constraints)
        prob.solve()

        if prob.status == 'optimal':
          return x.value
        else:
          return None


    def run(self, T):
        chosen_arms = np.zeros(T, dtype=int)

        for t in tqdm(range(T)):
            arm = self.select_arm(t)

            if t >= self.k and not self.is_uniform:
              self.perturbation = self.find_perturbation_with_l2_ball(arm, t)

            sample = self.logged_data[arm][int(self.N[arm])]
            self.N[arm] += 1
            self.empirical_means[arm] = self.empirical_means[arm] + (sample - self.empirical_means[arm])/self.N[arm]

            chosen_arms[t] = arm

        return chosen_arms

In [43]:
# check all inequalities (mode=1)
epsilon_attack = 0.5
qp = False
target_arm = 2
epsilon = 0.1
random.seed(42)
find_perturbation = FindPerturbationEpsilonGreedyAlgorithm(k, d, epsilon, target_arm, mu, logged_data, epsilon_attack, qp)
chosen_arms = find_perturbation.run(T)
print("\nNumber of pulls per arm:", find_perturbation.N)
print(chosen_arms)

100%|██████████| 50/50 [00:05<00:00,  8.38it/s]


Number of pulls per arm: [ 4.  3. 43.]
[0 1 2 2 1 2 2 2 2 2 2 0 2 2 0 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 1 2 2 2 2
 2 2 2 2 2 0 2 2 2 2 2 2 2]


In [44]:
perturbation = find_perturbation.perturbation
norm = np.linalg.norm(perturbation)
print(norm)

0.1484765080387139


In [45]:
# run epsilon-greedy with perturbation
epsilon = 0.1
random.seed(42)
epsilon_greedy = EpsilonGreedyAlgorithm(k, d, epsilon, mu, logged_data, perturbation)
rewards, chosen_arms = epsilon_greedy.run(T)
print("\nNumber of pulls per arm:", epsilon_greedy.N)
print(chosen_arms)

100%|██████████| 50/50 [00:00<00:00, 49391.24it/s]


Number of pulls per arm: [ 4.  3. 43.]
[0 1 2 2 1 2 2 2 2 2 2 0 2 2 0 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 1 2 2 2 2
 2 2 2 2 2 0 2 2 2 2 2 2 2]
